In [1]:
import pandas as pd
import numpy as np
import ast

df = pd.read_csv("../data/processed/jobs_match_scored.csv")

In [2]:
candidate = {
    "skills": {
        "python",
        "docker",
        "postgresql"
    },
    "experience_years": 3,
    "city": "تهران",
    "remote_preference": True,
    "target_roles": [
        "python",
        "backend",
        "machine learning",
        "هوش مصنوعی"
    ]
}

In [3]:
def ensure_set(x):
    if isinstance(x, set):
        return x
    if isinstance(x, str):
        if x == "{}" or x == "":
            return set()
        x = (x.replace("{", "").replace("}", ""))
        return set(item.strip() for item in x.split(","))
    return set()

df["required_skills_set"] = df["required_skills_set"].apply(ensure_set)

In [13]:
candidate_profile = {
    "skills": [
        "Python",
        "SQL",
        "Git",
        "Linux"
    ],
    "experience_years": 2,
    "city": "گرگان",
    "remote_preference": False,
    "target_roles": [
        "Data Analyst",
        "Business Intelligence",
        "BI Analyst"
    ]
}

candidate_skill_set = {skill.strip().lower() for skill in candidate_profile["skills"]}


# Matching Functions
def calculate_skill_coverage(candidate_skills: set, job_skills: set) -> float:
    """
    Percentage of required job skills covered by the candidate.

    Coverage = |Candidate Skills ∩ Job Skills| / |Job Skills|
    """
    if not job_skills:
        return 0.0

    matched = candidate_skills & job_skills
    return len(matched) / len(job_skills)


def calculate_skill_strength(candidate_skills: set, job_skills: set) -> float:
    """
    Percentage of the candidate's skills that are relevant to the job.

    Strength = |Candidate Skills ∩ Job Skills| / |Candidate Skills|
    """
    if not candidate_skills:
        return 0.0

    matched = candidate_skills & job_skills
    return len(matched) / len(candidate_skills)


def calculate_skill_complexity(job_skills: set) -> float:
    """
    Estimate job complexity based on the number of required skills.
    Maximum score is capped at 1.0.
    """
    if not job_skills:
        return 0.0

    return min(len(job_skills) / 5, 1.0)


def calculate_title_match(title: str, target_roles: list) -> float:
    """
    Measure how well the job title matches the candidate's target roles.
    """
    if not isinstance(title, str):
        return 0.0

    title = title.lower()

    score = sum(
        role.lower() in title
        for role in target_roles
    )

    return min(score / len(target_roles), 1.0)


def calculate_experience_match(candidate_exp: int, job_exp: int) -> float:
    """
    Compare candidate experience with required job experience.
    """
    difference = abs(candidate_exp - job_exp)
    return 1 / (1 + difference)


def calculate_location_match(candidate_city: str, job_city: str) -> float:
    """
    Exact city match.
    """
    if candidate_city == job_city:
        return 1.0

    return 0.0


def calculate_remote_match(candidate_remote: bool, job_remote: bool) -> float:
    """
    Candidate is willing to work onsite.
    """
    if not candidate_remote:
        return 1.0

    if job_remote:
        return 1.0

    return 0.3

In [39]:
def restore_skill_set(value):
    """
    Convert string representation of set back to Python set.
    """
    if isinstance(value, set):
        return value
    if isinstance(value, str):
        return ast.literal_eval(value)
    return set()

df["required_skills_set"] = df["required_skills_set"].apply(restore_skill_set)

In [40]:
candidate_skills = candidate["skills"]

df["skill_coverage_score"] = df["required_skills_set"].apply(lambda x: calculate_skill_coverage(candidate_skills, x))

df["skill_strength_score"] = df["required_skills_set"].apply(lambda x: calculate_skill_strength(candidate_skills, x))

df["skill_complexity_score"] = df["required_skills_set"].apply(calculate_skill_complexity)

df["title_match_score"] = df["title"].apply(lambda x: calculate_title_match(x, candidate["target_roles"]))

df["experience_match_score"] = df["experience_years"].apply(lambda x: calculate_experience_match(candidate["experience_years"], x))

df["location_match_score"] = df["city"].apply(lambda x: calculate_location_match(candidate["city"], x))

df["remote_match_score"] = df["remote"].apply(lambda x: calculate_remote_match(candidate["remote_preference"], x))


def calculate_final_score(row):
    return (
        0.25 * row["skill_coverage_score"]
        +
        0.15 * row["skill_strength_score"]
        +
        0.10 * row["skill_complexity_score"]
        +
        0.20 * row["title_match_score"]
        +
        0.15 * row["experience_match_score"]
        +
        0.10 * row["location_match_score"]
        +
        0.05 * row["remote_match_score"]
    )

In [18]:
df["final_match_score"] = df.apply(calculate_final_score, axis=1)

In [19]:
def clean_skill_set(skill_set):
    cleaned = set()
    for skill in skill_set:
        skill = (
            skill
            .replace("'", "")
            .replace('"', "")
            .strip()
            .lower()
        )
        cleaned.add(skill)
    return cleaned

df["required_skills_set"] = df["required_skills_set"].apply(clean_skill_set)

In [20]:
df["skill_coverage_score"] = df["required_skills_set"].apply(lambda x: calculate_skill_coverage(candidate_skills, x))
df["skill_strength_score"] = df["required_skills_set"].apply(lambda x: calculate_skill_strength(candidate_skills, x))

In [21]:
df["final_match_score"] = df.apply(calculate_final_score, axis=1)

In [22]:
recommendations = (
    df[
    [
        "title",
        "company",
        "skill_coverage_score",
        "skill_strength_score",
        "title_match_score",
        "final_match_score"
    ]
    ]
    .sort_values(
        "final_match_score",
        ascending=False
    )
    .head(20)
)

recommendations

,title,company,skill_coverage_score,skill_strength_score,title_match_score,final_match_score
1,Senior Backend Engineer(Python/FastAPI)- Shoraka,بیمه بازار,0.428571,1.000000,0.50,0.757143
96,مهندس ارشد هوش مصنوعی و بک‌اند,قهوه ارگن,0.500000,1.000000,0.25,0.725000
30,برنامه‌نویس Python/Odoo,گروه بازرگانی بی نی سی,0.600000,1.000000,0.25,0.715000
141,توسعه‌دهنده عامل هوش مصنوعی (AI Agent Developer),شرکت داده و اعتبار سنجی تجارت ایرانیان (داتا),1.000000,0.666667,0.25,0.705000
60,Senior Backend Developer (Python),شرکت توسعه فن آوری هوش مصنوعی تفاهم,0.333333,1.000000,0.50,0.698333
19,Senior Backend Developer (Python),گروه MCI NEXT,0.750000,1.000000,0.50,0.682500
17,برنامه‌نویس Back-End (Python),پارس تصمیم,0.428571,1.000000,0.25,0.672143
90,مهندس هوش مصنوعی و نرم‌افزار,نهادین آرمان,0.428571,1.000000,0.25,0.672143
168,کارشناس ارشد هوش مصنوعی (AI & Process Optimiza...,ایکس چیف,1.000000,0.333333,0.25,0.670000
8,توسعه‌دهنده Python,پارادایس هاب,0.375000,1.000000,0.25,0.658750


In [23]:
candidate_profile = {
    "skills": {
        "python",
        "docker",
        "postgresql"
    },
    "experience_years": 3,
    "city": "تهران",
    "remote_preference": True
}

In [24]:
def normalize_candidate_skills(skills):
    return {skill.lower().strip().replace(" ", "_") for skill in skills}

In [25]:
def calculate_skill_gap(candidate_skills, job_skills):
    missing = job_skills - candidate_skills
    return list(missing)

df["missing_skills"] = df["required_skills_set"].apply(lambda x: calculate_skill_gap(candidate_skills, x))

In [26]:
def recommend_jobs(candidate_profile, df, top_n=10):

    candidate_skills = normalize_candidate_skills(candidate_profile["skills"])

    result = df.copy()

    # Skill matching
    result["skill_coverage_score"] = result["required_skills_set"].apply(lambda x: calculate_skill_coverage(candidate_skills, x))

    result["skill_strength_score"] = result["required_skills_set"].apply(lambda x: calculate_skill_strength(candidate_skills, x))

    # Skill gap explanation
    result["missing_skills"] = result["required_skills_set"].apply(lambda x: calculate_skill_gap(candidate_skills, x))

    # Final ranking score
    result["final_match_score"] = result.apply(calculate_final_score, axis=1)

    return (
        result[
            [
                "title",
                "company",
                "skill_coverage_score",
                "skill_strength_score",
                "title_match_score",
                "final_match_score",
                "missing_skills"
            ]
        ]
        .sort_values(
            "final_match_score",
            ascending=False
        )
        .head(top_n)
    )

In [27]:
recommend_jobs(candidate_profile, df, top_n=20)

,title,company,skill_coverage_score,skill_strength_score,title_match_score,final_match_score,missing_skills
1,Senior Backend Engineer(Python/FastAPI)- Shoraka,بیمه بازار,0.428571,1.000000,0.50,0.757143,"[kubernetes, prometheus, redis, gerafana]"
96,مهندس ارشد هوش مصنوعی و بک‌اند,قهوه ارگن,0.500000,1.000000,0.25,0.725000,"[next.js, mongodb, git]"
30,برنامه‌نویس Python/Odoo,گروه بازرگانی بی نی سی,0.600000,1.000000,0.25,0.715000,"[javascript, git]"
141,توسعه‌دهنده عامل هوش مصنوعی (AI Agent Developer),شرکت داده و اعتبار سنجی تجارت ایرانیان (داتا),1.000000,0.666667,0.25,0.705000,[]
60,Senior Backend Developer (Python),شرکت توسعه فن آوری هوش مصنوعی تفاهم,0.333333,1.000000,0.50,0.698333,"[prometheus, django, redis, git, rest_api, ger..."
19,Senior Backend Developer (Python),گروه MCI NEXT,0.750000,1.000000,0.50,0.682500,[django]
17,برنامه‌نویس Back-End (Python),پارس تصمیم,0.428571,1.000000,0.25,0.672143,"[rest_api, mongodb, rabbitmq, git]"
90,مهندس هوش مصنوعی و نرم‌افزار,نهادین آرمان,0.428571,1.000000,0.25,0.672143,"[rest_api, django, mysql, git]"
168,کارشناس ارشد هوش مصنوعی (AI & Process Optimiza...,ایکس چیف,1.000000,0.333333,0.25,0.670000,[]
8,توسعه‌دهنده Python,پارادایس هاب,0.375000,1.000000,0.25,0.658750,"[mongodb, django, mysql, redis, git]"


In [28]:
class JobMatcher:

    def __init__(self, candidate):
        self.candidate = candidate
        self.candidate_skills = normalize_candidate_skills(candidate["skills"])


    def score_jobs(self, df):
        result = df.copy()
        result["skill_coverage_score"] = result["required_skills_set"].apply(lambda x: calculate_skill_coverage(self.candidate_skills, x))

        result["skill_strength_score"] = result["required_skills_set"].apply(lambda x: calculate_skill_strength(self.candidate_skills, x))

        result["missing_skills"] = result["required_skills_set"].apply(lambda x: calculate_skill_gap(self.candidate_skills, x))

        result["final_match_score"] = result.apply(calculate_final_score, axis=1)
        return result


    def rank_jobs(self, df, top_n=20):

        scored_jobs = self.score_jobs(df)


        return (
            scored_jobs[
                [
                    "title",
                    "company",
                    "skill_coverage_score",
                    "skill_strength_score",
                    "title_match_score",
                    "final_match_score",
                    "missing_skills"
                ]
            ]
            .sort_values(
                "final_match_score",
                ascending=False
            )
            .head(top_n)
        )

In [32]:
candidate_backend = {
    "skills": [
        "Python",
        "Docker",
        "PostgreSQL",
        "Git",
        "Django",
        "Redis"
    ],
    "experience_years": 4,
    "city": "تهران",
    "remote_preference": True
}


backend_results = recommend_jobs(
    candidate_backend,
    df,
    top_n=10
)

backend_results

# *******************************************************************************************************************************

candidate_data = {
    "skills": [
        "Python",
        "SQL",
        "Power BI",
        "Pandas",
        "Machine Learning",
        "Statistics"
    ],
    "experience_years": 3,
    "city": "تهران",
    "remote_preference": True
}


data_results = recommend_jobs(
    candidate_data,
    df,
    top_n=10
)

data_results

# *******************************************************************************************************************************

candidate_devops = {
    "skills": [
        "Linux",
        "Docker",
        "Kubernetes",
        "Ansible",
        "Git",
        "Prometheus"
    ],
    "experience_years": 5,
    "city": "تهران",
    "remote_preference": True
}

# *******************************************************************************************************************************

me_per_se = {
    "skills": [
        "Python",
        "SQL",
        "Git",
        "Linux"
    ],
    "experience_years": 2,
    "city": "گرگان",
    "remote_preference": False,
    "target_roles": [
        "Data Analyst",
        "Business Intelligence",
        "BI Analyst"
    ]
}

In [33]:
backend_results = JobMatcher(candidate_backend).rank_jobs(df, 10)
data_results = JobMatcher(candidate_data).rank_jobs(df, 10)
devops_results = JobMatcher(candidate_devops).rank_jobs(df, 10)
me = JobMatcher(me_per_se).rank_jobs(df, 20)

In [ ]:
devops_results

,title,company,skill_coverage_score,skill_strength_score,title_match_score,final_match_score,missing_skills
506,کارشناس DevOps,سازمانی فعال در حوزه سرمایه گذاری و مالی,0.857143,1.000000,0.00,0.729286,[jenkins]
1,Senior Backend Engineer(Python/FastAPI)- Shoraka,بیمه بازار,0.428571,0.500000,0.50,0.682143,"[postgresql, python, redis, gerafana]"
440,DevOps Engineer,اسکای روم,0.625000,0.833333,0.00,0.681250,"[postgresql, gitlab, gerafana]"
479,DevOps Engineer,تجارت الکترونیک ستون,0.714286,0.833333,0.00,0.668571,"[gitlab, gerafana]"
493,DevOps Engineer,شرکت تدبیر افزار پشتیبان تک (تاپ تک),0.714286,0.833333,0.00,0.668571,"[python, go]"
1351,کارشناس DevOps,ایران فاوا گسترش,0.714286,0.833333,0.00,0.668571,"[helm, gerafana]"
132,Kubernetes SRE Engineer,سرورایانه,0.714286,0.833333,0.00,0.668571,"[python, power_bi]"
283,کارشناس DevOps,داتین,0.714286,0.833333,0.00,0.668571,"[jenkins, gitlab]"
28,توسعه‌دهنده تست خودکار (Python),نوآوران امن اندیش شریف,0.750000,0.500000,0.25,0.657500,[python]
515,Senior Monitoring & Observability Engineer,مهسان,1.000000,0.500000,0.00,0.650000,[]


In [37]:
recommendation_df = df[
[
"title",
"company",
"city",
"required_skills_set",
"experience_years",
"is_remote",
"salary_avg",
"skill_level_score",
"skill_coverage_score",
"skill_strength_score",
"title_match_score",
"experience_match_score",
"location_match_score",
"remote_match_score",
"final_match_score",
"missing_skills"
]
]
recommendation_df.to_csv("../data/processed/jobs_recommendation.csv", index=False)

In [38]:
dashboard_df = df[
[
"title",
"company",
"city",
"province",
"category",
"industries_clean",
"benefits_clean",
"experience_level",
"seniority",
"company_size",
"salary_avg",
"application_count",
"required_skills_set"
]
]
dashboard_df.to_parquet("../data/processed/jobs_dashboard.csv", index=False)